## Imports

In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import matplotlib.mlab   as mlab

from xkte_nonadaptive import kernel_dr_two_sample_test_agnostic
from projected_adaptive_kte import projected_adaptive_kte_test
from baselines import cadr_test, hadad_test, fit_krr_predict
from sklearn.metrics import pairwise_distances

from scipy.spatial.distance import cdist
from scipy.special import expit
from scipy.stats import bernoulli
from numpy.polynomial.polynomial import polyval
from sklearn.linear_model import LogisticRegression

from scipy.stats import norm
import scipy.stats as stats
import statistics

from tqdm import tqdm

import seaborn as sns
import pandas as pd
import time
import os


In [2]:
import numpy as np

def scenario_signal_vector(ns, scenario, rng, beta_mix=2.0, beta_uniform=4.0):
    """
    Scenario-driven *scalar* multipliers for the heterogeneous TE direction u.

    Output: vector g ∈ R^ns such that Δ_t = g_t * (X_t @ u)
    """
    if scenario == 'I':   # no treatment effect (H0)
        return np.zeros(ns)

    if scenario == 'II':  # constant magnitude (positive)
        return np.full(ns, 2.0)

    if scenario == 'III':  # random sign * fixed magnitude
        signs = rng.binomial(1, 0.5, ns) * 2 - 1
        return signs.astype(float) * beta_mix

    if scenario == 'IV':  # continuous heterogeneous magnitude
        return rng.uniform(-beta_uniform, beta_uniform, ns)

    return np.zeros(ns)


def make_data_with_scenarios(n, d, rng, sigma_eps=1.0, scenario='I'):
    """
    Linear baseline + scenario-based heterogeneous TE along direction u.
    Δ_t = g_t * (X_t @ u)
    """

    # Gaussian covariates
    X = rng.normal(size=(n, d))

    # unit directions
    v = rng.normal(size=d)
    v /= np.linalg.norm(v) + 1e-12

    u = rng.normal(size=d)
    u /= np.linalg.norm(u) + 1e-12

    # baseline f(x)
    f = X @ v

    # scenario-based TE multipliers
    g = scenario_signal_vector(n, scenario, rng)

    # heterogeneous TE
    Delta = g * (X @ u)

    m0 = f
    m1 = f + Delta

    eps = rng.normal(scale=sigma_eps, size=n)

    return X, m0, m1, Delta, eps, u



def policy_explore_commit(x, t, t0, eps, u, winner):
    """
    Epsilon explore-then-commit with linear separator on x @ u.

    parameters
    ----------
    x : (d,) array
    t : int
    t0 : int
    eps : float
    u : (d,) array
    winner : int in {0,1}

    returns
    -------
    p1 : float
        Probability of action 1 at time t given context x.
    """
    if t < t0:
        return 0.5
    inS = (x @ u) > 0
    prefer1 = (winner == 1 and inS) or (winner == 0 and not inS)
    return (1.0 - eps) if prefer1 else eps


def generate_adaptive_data(
    ns,
    d=5,
    sigma_eps=1.0,
    scenario='I',
    t0=15,
    eps=0.001,
    rng=None,
    split="alternating",
):
    rng = rng or np.random.RandomState(0)

    # NEW: scenario-based DGP
    X, m0, m1, Delta, eps_noise, u = make_data_with_scenarios(
        n=ns,
        d=d,
        rng=rng,
        sigma_eps=sigma_eps,
        scenario=scenario,
    )

    # ETC winner
    winner = 1 if rng.rand() < 0.5 else 0

    # predictable propensities
    w = np.array(
        [policy_explore_commit(X[t], t, t0, eps, u, winner) for t in range(ns)]
    )

    # realized actions + outcomes
    A = (rng.rand(ns) < w).astype(int)
    Y = np.where(A == 1, m1, m0) + eps_noise


    # ------------------ folds for cross-evaluation ------------------ #
    if split == "alternating":
        idx0 = np.arange(0, ns, 2)
        idx1 = np.arange(1, ns, 2)
    else:  # chronological
        cut = ns // 2
        idx0 = np.arange(0, cut)
        idx1 = np.arange(cut, ns)

    X0 = X[idx0]
    X1 = X[idx1]

    N0, N1 = len(idx0), len(idx1)

    # For each t in fold0, compute policy probs on all X0
    Pi_fold0_on_0 = np.empty((N0, N0), dtype=float)
    for r, t in enumerate(idx0):
        Pi_fold0_on_0[r] = np.array(
            [policy_explore_commit(x, t, t0, eps, u, winner) for x in X0]
        )

    # For each t in fold1, compute policy probs on all X1
    Pi_fold1_on_1 = np.empty((N1, N1), dtype=float)
    for r, t in enumerate(idx1):
        Pi_fold1_on_1[r] = np.array(
            [policy_explore_commit(x, t, t0, eps, u, winner) for x in X1]
        )

    # Full matrix: for each t, policy probs on all X
    P_all = np.empty((ns, ns), dtype=np.float32)
    for t in range(ns):
        P_all[t] = np.array(
            [policy_explore_commit(x, t, t0, eps, u, winner) for x in X]
        )

    return X, A, Y[:, None], w, Pi_fold0_on_0, Pi_fold1_on_1, idx0, idx1, P_all


In [3]:
import numpy as np

def treatment_effect_vector(ns, scenario, rng, beta_mix=2.0, beta_uniform=4.0):
    if scenario == 'I':
        return np.zeros(ns)
    if scenario == 'II':
        return np.full(ns, 2.0)
    if scenario == 'III':
        signs = rng.binomial(1, 0.5, ns) * 2 - 1
        return signs.astype(float) * beta_mix
    if scenario == 'IV':
        return rng.uniform(-beta_uniform, beta_uniform, ns)
    return np.zeros(ns)

h_sigmoidal = lambda x: np.log(np.abs(16 * x - 8) + 1) * np.sign(x - 0.5)

def generate_adaptive_data(
    ns, d, beta_vec, noise_var, scenario,
    eps0=0.2, eps_min=0.01, power=0.99, lam=1e-2,
    rng=None, split="alternating"
):
    rng = rng or np.random.RandomState(0)

    X = rng.randn(ns, d)
    base = h_sigmoidal(X @ beta_vec)
    delta = treatment_effect_vector(ns, scenario, rng)

    Y0 = base + noise_var * rng.randn(ns)
    Y1 = base + noise_var * rng.randn(ns) + delta

    X_aug = np.hstack([np.ones((ns, 1)), X])

    S0 = np.diag([0.0] + [lam] * d)
    S1 = np.diag([0.0] + [lam] * d)
    b0 = np.zeros(d + 1)
    b1 = np.zeros(d + 1)

    def solve(S, b):
        try:
            return np.linalg.solve(S, b)
        except:
            return np.linalg.lstsq(S, b, rcond=None)[0]

    def eps_at(t):
        return max(eps_min, eps0 / ((t + 1)**power))

    A = np.zeros(ns, dtype=int)
    w = np.zeros(ns)
    Y = np.zeros(ns)

    theta0_snap = np.zeros((ns, d+1))
    theta1_snap = np.zeros((ns, d+1))

    for t in range(ns):
        th0 = solve(S0, b0)
        th1 = solve(S1, b1)
        theta0_snap[t] = th0
        theta1_snap[t] = th1

        eps_t = eps_at(t)
        z = X_aug[t]

        q0 = z @ th0
        q1 = z @ th1

        if q1 > q0:
            pi1 = 1 - 0.5 * eps_t
        elif q1 < q0:
            pi1 = 0.5 * eps_t
        else:
            pi1 = 0.5

        a = 1 if rng.rand() < pi1 else 0
        A[t] = a
        w[t] = pi1
        Y[t] = Y1[t] if a == 1 else Y0[t]

        if a == 0:
            S0 += np.outer(z, z)
            b0 += z * Y[t]
        else:
            S1 += np.outer(z, z)
            b1 += z * Y[t]

    if split == "alternating":
        idx0 = np.arange(0, ns, 2)
        idx1 = np.arange(1, ns, 2)
    else:
        cut = ns // 2
        idx0 = np.arange(0, cut)
        idx1 = np.arange(cut, ns)

    Z0 = X_aug[idx0]
    Z1 = X_aug[idx1]

    N0, N1 = len(idx0), len(idx1)

    Pi_fold0_on_0 = np.empty((N0, N0))
    for r, t in enumerate(idx0):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0 = Z0 @ th0; q1 = Z0 @ th1
        Pi_fold0_on_0[r] = np.where(
            q1 > q0, 1 - 0.5 * eps_t,
            np.where(q1 < q0, 0.5 * eps_t, 0.5)
        )

    Pi_fold1_on_1 = np.empty((N1, N1))
    for r, t in enumerate(idx1):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0 = Z1 @ th0; q1 = Z1 @ th1
        Pi_fold1_on_1[r] = np.where(
            q1 > q0, 1 - 0.5 * eps_t,
            np.where(q1 < q0, 0.5 * eps_t, 0.5)
        )

    P_all = np.empty((ns, ns), dtype=np.float32)
    for t in range(ns):
        th0 = theta0_snap[t]; th1 = theta1_snap[t]; eps_t = eps_at(t)
        q0_all = X_aug @ th0
        q1_all = X_aug @ th1
        P_all[t] = np.where(
            q1_all > q0_all, 1 - 0.5 * eps_t,
            np.where(q1_all < q0_all, 0.5 * eps_t, 0.5)
        )

    return X, A, Y[:, None], w, Pi_fold0_on_0, Pi_fold1_on_1, idx0, idx1, P_all


## Main function, which runs a list of tests based on its arguments

In [9]:
import os, time
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

def run_tests_adaptive(
    b_list, method_list, ns_list,
    name_folder, num_experiments, iterations,
    seed=42,
    split="alternating"   # 'alternating' or 'chronological'
):
    if split not in ("alternating", "chronological"):
        raise ValueError("split must be 'alternating' or 'chronological'.")

    noise_var = 0.5
    d = 5
    beta_vec = np.array([0.1, 0.2, 0.3, 0.4, 0.5])
    os.makedirs(name_folder, exist_ok=True)

    rng = np.random.RandomState(seed)

    for b in b_list:
        print("b =", b)
        for method in method_list:
            for ns in tqdm(ns_list):
                p_values = np.zeros(num_experiments)
                values   = np.zeros(num_experiments)
                times    = np.zeros(num_experiments)

                for n in range(num_experiments):
                    X, T, Y, w, Pi_0_on_0, Pi_1_on_1, idx0, idx1, P_all = generate_adaptive_data(
                        ns=ns,
                        d=d,
                        beta_vec=beta_vec,
                        noise_var=noise_var,
                        scenario=b,
                        eps0=0.5,
                        eps_min=0.005,
                        power=0.25,
                        lam=1e-2,
                        rng=rng,
                        split=split,
                    )

                    if Y.ndim == 1:
                        Y = Y[:, None]

                    YY0 = Y[T == 0]
                    YY1 = Y[T == 1]
                    sigma2 = np.median(
                        pairwise_distances(YY0, YY1, metric="euclidean")
                    )**2 / 4
                    if (not np.isfinite(sigma2)) or sigma2 <= 0:
                        sigma2 = float(np.var(Y)) + 1e-6

                    y = Y.reshape(-1)
                    t0 = time.time()

                    if method == "ADR-KTE":
                        out = projected_adaptive_kte_test(
                            Y=Y,
                            X=X,
                            A=T,
                            policy_matrix=P_all,
                            idx_pilot=idx0,
                            idx_infer=idx1,
                            outcome_gamma=1.0/sigma2,
                            ridge=5*1e-2,
                        )
                        value = out["statistic"]
                        p_value = out["p_value"]

                    elif method == "DR-xKTE":
                        value, p_value = kernel_dr_two_sample_test_agnostic(
                            Y, X, T, w,
                            kernel_function='rbf',
                            lam=5*1e-2,
                            gamma=1.0/sigma2,
                            verbose=False
                        )
                        from math import erf
                        p_value = 0.5 * (1.0 - erf(value / np.sqrt(2.0)))

                    # elif method == "KTE":
                    #     stat, _, pval = kernel_two_sample_test_nonuniform(
                    #         YY0,
                    #         YY1,
                    #         T,
                    #         w.reshape(-1),
                    #         kernel_function="rbf",
                    #         iterations=iterations,
                    #         random_state=seed,
                    #         gamma=1.0 / sigma2,
                    #     )
                    #     value = stat
                    #     p_value = pval

                    # elif method == 'DR-CFME':
                    #     t0 = time.time()
                    #     stat, _, p_value = kernel_dr_nonuniform(Y, X, T, w, experiment=False, iterations=10, verbose=False, kernel_function='rbf', gamma=1.0/sigma2)
                    #     value=stat
                    
                    else:
                        raise ValueError("Method not recognized.")

                    times[n]   = time.time() - t0
                    p_values[n] = p_value
                    values[n]   = value

                df = pd.DataFrame(
                    {
                        "times": times,
                        "p_values": p_values,
                        "stat_values": values,
                    }
                )
                df.to_csv(
                    os.path.join(name_folder, f"ns{ns}b{b}{method}_{split}.csv"),
                    index=False,
                )


## Run functions for different settings

### Null hypothesis

In [10]:
num_experiments=200
iterations=100

ns_list = np.arange(100, 550, 50)
b_list = ['I']
# method_list = ['DR-xKTE']
# method_list = ['ADR-KTE', 'DR-xKTE', 'KTE']
method_list = ['ADR-KTE']


experiment = 'adaptive_rebuttal'
name_folder = 'results/' + str(experiment) + '/'
run_tests_adaptive(b_list, method_list, ns_list, name_folder, num_experiments, iterations, split="alternating")

b = I


100%|██████████| 9/9 [01:19<00:00,  8.87s/it]


## H1 scenarios II, III, IV

In [11]:
num_experiments=200
iterations=100

ns_list = np.arange(100, 550, 50)
b_list = ['II']
# method_list = ['ADR-KTE', 'DR-xKTE', 'KTE']
# method_list = ['DR-xKTE']
method_list = ['ADR-KTE']


experiment = 'adaptive_rebuttal'
name_folder = 'results/' +str(experiment) + '/'
run_tests_adaptive(b_list, method_list, ns_list, name_folder, num_experiments, iterations, split="alternating")

b = II


100%|██████████| 9/9 [01:43<00:00, 11.46s/it]


In [12]:
import os
import numpy as np
import pandas as pd

def aggregate_results_adaptive(
    name_folder="results/adaptive_rebuttal/",
    scenario_list=("I", "II"),
    ns_list_null=np.arange(100, 550, 50),   # from cell 7
    ns_list_alt=np.arange(100, 550, 50),    # from cell 12
    methods=("ADR-KTE", "DR-xKTE"),
    split_suffix="_alternating.csv",
    alpha=0.05,
):
    rows = []

    # Scenario I (null) uses ns_list_null; II–IV use ns_list_alt
    for scenario in scenario_list:
        if scenario == "I":
            ns_list = ns_list_null
        else:
            ns_list = ns_list_alt

        for method in methods:
            for ns in ns_list:
                fname = f"{name_folder}ns{ns}b{scenario}{method}{split_suffix}"
                if not os.path.exists(fname):
                    # skip missing files
                    continue

                df = pd.read_csv(fname)
                pvals = df["p_values"].values
                stat_vals = df["stat_values"].values
                times = df["times"].values

                rej = (pvals < alpha).mean()
                rows.append(
                    {
                        "scenario": scenario,
                        "ns": ns,
                        "method": method,
                        "rej_rate": rej,
                        "mean_stat": stat_vals.mean(),
                        "mean_time": times.mean(),
                        "n_experiments": len(pvals),
                    }
                )

    results_df = pd.DataFrame(rows)
    return results_df

results_table = aggregate_results_adaptive()
results_table

results_pivot = (
    results_table
    .pivot_table(
        index=["scenario", "ns"],
        columns="method",
        values="rej_rate",
    )
    .reset_index()
)


In [13]:
# --- Build the master results table ---
results_table = aggregate_results_adaptive()

# --- Produce one transposed table per scenario ---
scenario_tables = {}

for scenario in results_table["scenario"].unique():
    df_s = results_table[results_table["scenario"] == scenario]

    # Pivot: rows = methods, columns = ns
    table = (
        df_s.pivot_table(
            index="method",
            columns="ns",
            values="rej_rate",
        )
        .sort_index(axis=1)
        .sort_index(axis=0)
    )

    scenario_tables[scenario] = table

# Display all scenario-specific tables
for scenario, table in scenario_tables.items():
    print(f"\n===== Scenario {scenario} =====\n")
    display(table)



===== Scenario I =====



ns,100,150,200,250,300,350,400,450,500
method,,,,,,,,,
DR-xKTE,0.165,0.150,0.125,0.10,0.080,0.090,0.105,0.095,0.075
ADR-KTE,0.140,0.125,0.125,0.12,0.145,0.115,0.065,0.095,0.060



===== Scenario II =====



ns,100,150,200,250,300,350,400,450,500
method,,,,,,,,,
DR-xKTE,0.960,1.000,0.985,0.995,1.00,1.000,1.0,1.0,1.0
ADR-KTE,0.885,0.945,0.950,0.955,0.99,0.995,1.0,1.0,1.0
